In [1]:
from pytential.sympy_pytential import sympy_pytential
import numpy as np
import matplotlib.pyplot as plt
from pytential.reduce.matrix_methods import reduce_qp

Create two ideal mixing functions from a set of properties, and check them.

In [2]:
from sympy import log, symbols
c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd = symbols('c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd')

In [3]:
T = 1600
RT = 8.134*T
mu0_SiC = -161028
rho_SiC = 3.21 / 40.11 * 1e6
rho_Ar = 101e3 / RT
v_SiC = 1/rho_SiC
v_Ar = 1/rho_Ar

In [4]:
fa_sp = c0a*mu0_SiC 
fb_sp = c0b*mu0_SiC
fc_sp = c0c*mu0_SiC 
fd_sp = c0d*(-120100+RT*log(c0d/(c0d+c1d))) +c1d*(-290457+RT*log(c1d/(c0d+c1d)))

In [5]:
np.exp((mu0_SiC--120100)/RT)

0.04307449610427493

In [6]:
fa = sympy_pytential(fa_sp, constraints_sym=[c0a-Va])
fb = sympy_pytential(fb_sp, constraints_sym=[c0b-Vb])
fc = sympy_pytential(fc_sp, constraints_sym=[c0c-Vc])
fd = sympy_pytential(fd_sp, constraints_sym=[c0d+c1d-Vd])

Make a function fa+fb with all the variables, and add constraints that the concentrations must sum to ca and cb

In [7]:
f = fa+fb+fc+fd
f = f.add_constraints_sym([c0a+c0b+c0c+c0d-c0, c1d-c1])
print(f)


Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + c0d*(13014.4*log(c0d/(c0d + c1d)) - 120100) + c1d*(13014.4*log(c1d/(c0d + c1d)) - 290457)

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 13014.4*log(c0d/(c0d + c1d)) - 120100.0, 0, 13014.4*log(c1d/(c0d + c1d)) - 290457.0]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 13014.4*c1d/(c0d*(c0d + c1d)), 0, -13014.4/(c0d + c1d)], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -13014.4/(c0d + c1d), 0, 13014.4*c0d/(c1d*(c0d + c1d))]]

Constraints
-Va + c0a
-Vb + c0b
-Vc + c0c
-Vd + c0d + c1d
-c0 + c0a + c0b + c0c + c0d
-c1 + c1d



In [8]:
A = np.array(f.get_constraint_jacobian(), dtype = np.float64)
print(A.shape)
np.linalg.inv(A@A.T)

(6, 11)


array([[ 0.58064516,  0.08064516,  0.08064516,  0.06451613, -0.16129032,
        -0.03225806],
       [ 0.08064516,  0.58064516,  0.08064516,  0.06451613, -0.16129032,
        -0.03225806],
       [ 0.08064516,  0.08064516,  0.58064516,  0.06451613, -0.16129032,
        -0.03225806],
       [ 0.06451613,  0.06451613,  0.06451613,  0.4516129 , -0.12903226,
        -0.22580645],
       [-0.16129032, -0.16129032, -0.16129032, -0.12903226,  0.32258065,
         0.06451613],
       [-0.03225806, -0.03225806, -0.03225806, -0.22580645,  0.06451613,
         0.61290323]])

In [9]:
y0 = {'Va':1, 'Vb':1, 'Vc':1, 'Vd':1, 'c0a':rho_SiC, 'c0b':rho_SiC, 'c0c':rho_SiC , 'c0d':rho_Ar*0.043, 'c1d':rho_Ar*(1-0.043), 'c0':3*rho_SiC+rho_Ar*.043,'c1':rho_Ar*(1-.043)}

In [10]:
fq = f.quadratic_expansion(y0)
print(fq)
print(fq.vars)
print(fq.grad(**y0))
print(fq.get_constraint_jacobian())



Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + 0.5*c0d*(37322.4727707852*c0d - 1676.97631049505*c1d) - 161050.527517103*c0d + 0.5*c1d*(-1676.97631049505*c0d + 75.3500327599657*c1d) - 291029.00744506*c1d - 38663387969.8234

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 37322.4727707852*c0d - 1676.97631049505*c1d - 161050.527517103, 0, -1676.97631049505*c0d + 75.3500327599657*c1d - 291029.00744506]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 37322.4727707852, 0, -1676.97631049505], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -1676.97631049505, 0, 75.3500327599657]]

Constraints
-Va + c0a
-Vb + c0b
-Vc + c0c
-Vd + c

In [11]:
print(fq.vars)
fr, dep_expr = fq.remove_linear_constraints(['c0', 'c1', 'Va', 'Vb', 'Vc', 'Vd'], y0=y0)
#print(fr)
#print(fr.vars)
#print('grad', fr.grad(**y0))

['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']
free_indices [4, 9, 0, 1, 2, 3]
A_d has condition number 3.235465634722064, rank 5, and dimensions (6, 5)
Q_tilde is symmetric.
old function
 [[  4335.36517661  -8956.50557436  -4335.36517661  -4335.36517661
   -4335.36517661   8384.95513207]
 [  4335.36517661  -8956.50557436  -4335.36517661  -4335.36517661
   -4335.36517661   8384.95513207]
 [  4335.36517661  -8956.50557436  -4335.36517661  -4335.36517661
   -4335.36517661   8384.95513207]
 [ -8384.95513207  17322.62319824   8384.95513207   8384.95513207
    8384.95513207 -16217.19733004]
 [ -4335.36517661   8956.50557436   4335.36517661   4335.36517661
    4335.36517661  -8384.95513207]
 [  8956.50557436 -18503.3990992   -8956.50557436  -8956.50557436
   -8956.50557436  17322.62319824]] [ 75335.32540181  75335.32540181  75335.32540181  75357.85291891
  85692.67459819 215671.15452615]
sol [ 8.00299177e+04  2.87718789e-11 -1.14243324e-10  4.86890101e-11
 -3.8795078

sol [ 8.00299177e+04  2.87718789e-11 -1.14243324e-10  4.86890101e-11
 -3.87950783e-11] lam [ 75335.32540227  75335.32540227  75335.32540227  75357.85291799
  85692.67459773 215671.15452714]


sol [81363.74968835  1333.83196211  1333.8319621   2667.66392422
 -1333.8319621 ] lam [ 34771227.16522209  34771227.16522209  34771227.16522209
 -67029369.08310128 -34610199.1652221   71894511.57786204]

In [12]:
y1 = {'c0': 80029.9177262528, 'c1': 0, 'Va': .9, 'Vb': 0, 'Vc': 0, 'Vd': 0}
print(fr(**y1))
print(fr.grad(**y1))
dep_expr(y1)

13889575992797.89
[347034253.72258544, -184845388.9332796, 71433.49674297115, -508925913.70912576, -346873225.72258544, 694133250.7511944]


ValueError: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

In [ ]:
lam_lin = np.array([[-6.18428344e-24, -7.03045332e-07,  4.88411584e-18, -3.57416605e-18,
   5.22146643e-06],
 [ 1.16589051e-24,  1.32541448e-07, -9.20776737e-19,  6.73818775e-19,
  -9.84375673e-07],
 [ 2.18412391e-28,  2.48296854e-11, -1.72493941e-22,  1.26230009e-22,
  -1.84408263e-10],
 [ 2.66232918e-12,  3.02660465e+05, -2.10260805e-06,  1.53867569e-06,
  -2.24783721e+06],
 [ 6.17553483e-29,  7.02050768e-12, -4.87720652e-23,  3.56910985e-23,
  -5.21407987e-11],
 [-3.58469814e-13, -4.07517754e+04,  2.83106057e-07, -2.07175278e-07,
   3.02660465e+05]])
lam_const = [-6.47824565e-07, -4.89348047e-06,  6.71067203e-11,  1.74827824e+02,
  1.61028000e+05,  2.91006480e+05]
lam_lin@np.array([80029.9177262528, 0, 1, 0, 0])+lam_const

array([-6.47824565e-07, -4.89348047e-06,  6.71067203e-11,  1.74827822e+02,
        1.61028000e+05,  2.91006480e+05])

In [ ]:
[1.8783331795833331, -398267.9210385821, -95.39321080635419, 77.5576522980707, 160930.72964178462, 3248927.2350904713]

[1.8783331795833331,
 -398267.9210385821,
 -95.39321080635419,
 77.5576522980707,
 160930.72964178462,
 3248927.2350904713]